# Pelajaran 08 - Corak Reka Bentuk Pelbagai Ejen


## Persediaan


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mengapa Sistem Pelbagai Ejen?

Tugas dunia sebenar seperti perancangan perjalanan melibatkan pelbagai jenis kepakaran — logistik, pengetahuan tempatan, penganggaran, dan lain-lain. Seorang ejen tunggal yang cuba mengendalikan semuanya dengan cepat menjadi sukar diurus.

Sistem pelbagai ejen menyelesaikan ini melalui **pengkhususan**: setiap ejen menumpukan pada satu bidang kepakaran, menghasilkan hasil yang lebih berkualiti berbanding seorang generalis. Ia juga meningkatkan **kebolehskalaan** — anda boleh menambah ejen baru (contohnya, seorang pakar penerbangan, seorang pengkritik restoran) tanpa menulis semula aliran kerja sedia ada. Ejen-ejen ini digabungkan melalui rangkaian terstruktur, menyerahkan konteks dari satu ke yang lain.


## Mewujudkan Ejen Pakar


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Membangun Aliran Kerja Berurutan

`WorkflowBuilder` membolehkan anda menghubungkan ejen-ejen ke dalam graf berarah. Di sini kita mencipta saluran dua langkah yang ringkas: **TravelPlanner** merangka jadual perjalanan, kemudian **TravelConcierge** menyemak dan memperbaikinya.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Menambah Lebih Banyak Ejen ke Aliran Kerja

Salah satu kelebihan terbesar corak multi-ejen adalah betapa mudahnya untuk diperluas. Di bawah kami menambah ejen **BudgetReviewer** yang memeriksa rancangan terhadap bajet pengembara, menandakan item yang mungkin melebihi had kos, dan mencadangkan alternatif penjimatan wang. Aliran kerja kini menjalankan tiga ejen secara berurutan:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

In this lesson you learned how to:

1. **Create specialized agents** — each with a focused role (planning, concierge, budget review).
2. **Wire agents into a sequential workflow** using `WorkflowBuilder` and `add_edge`.
3. **Stream output** from a multi-agent pipeline, tracking which agent is speaking.
4. **Extend a workflow** by adding new agents to the chain without modifying existing ones.

The multi-agent design pattern keeps each agent simple while producing richer, more thoroughly reviewed results than any single agent could achieve alone.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
